# Spatial quantification and plotting

These demos target **movement 0.17.0** and one explicitly selected session.
`data-conduit` selects, reads and aligns the experiment's DLC and event streams;
`movement` filters the pose and calculates the measurements. Matplotlib arranges
panels and the shared template adds event labels.

The time axis remains the recorded session clock in **seconds**. Spatial values
remain **pixels**; no arena calibration is inferred. Filtering settings are
examples to review for this session, not validated paper analysis settings.
Processing runs on the continuous session before any trial is sliced.
No real-data output has been generated in this template.

In [ ]:
from pathlib import Path
import sys

# Locate this checkout when Jupyter starts in a notebook subdirectory.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "movement_figures").is_dir():
        sys.path.insert(0, str(parent))
        break
    if (parent / "src" / "movement_figures").is_dir():
        sys.path.insert(0, str(parent / "src"))
        break
else:
    raise RuntimeError("Start Jupyter inside the data-conduit checkout.")

import matplotlib.pyplot as plt
import movement
import movement.kinematics as kin
import numpy as np
import pandas as pd
from IPython.display import display
from movement.plots import plot_centroid_trajectory, plot_occupancy
from movement.utils.vector import compute_signed_angle_2d

from movement_figures.data_template.loading import (
    build_config, load_session, prepare_pose, preview_session,
)
from movement_figures.timeseries_template.annotations import annotate_qc_timeseries, qc_annotations

print(f"movement {movement.__version__}: {movement.__file__}")

## Session and processing settings

In [ ]:
# Edit this cell for your data. The same settings apply to all three notebooks.
ROOT = Path("/path/to/Training")
SESSION = "replace-with-one-session-folder-name"
LEVEL_NAMES = ("mouseID", "day")  # ROOT / mouseID / day / session
LEVEL_SELECTORS = None  # e.g. {"l0_selector": ["your-mouse"]}
SOURCES = ("trials", "events", "dlc")

INDIVIDUAL = "individual_0"
TRACKING_KEYPOINT = "body"  # one tracked point, used consistently for paths/speed/occupancy
LEFT_EAR, RIGHT_EAR = "lear", "rear"
BODY_FRONT, BODY_BACK = "body", "tailbase"
CAMERA_VIEW = "top_down"  # DLC image coordinates: +x right, +y down
REFERENCE_VECTOR = (1.0, 0.0)  # fixed axis; register to arena axes if different

CONFIDENCE_THRESHOLD = 0.9  # example setting: inspect your own confidence distribution
MAX_GAP_FRAMES = None  # None disables interpolation; an integer fills only short gaps
SMOOTHING_WINDOW = None  # None disables movement's rolling median filter

TRIAL_ROWS = (0, 1, 2)  # zero-based rows of the displayed, time-sorted session trial table
WINDOW = None  # absolute seconds (start, end); None shows the first 60 seconds of trials
OCCUPANCY_BINS = 40
ARENA_RANGE = None  # optionally ((xmin, xmax), (ymin, ymax)), in original pixels

config = build_config(
    ROOT, session=SESSION, level_names=LEVEL_NAMES,
    level_selectors=LEVEL_SELECTORS, sources=SOURCES,
)

## Load, inspect coverage, and process

In [ ]:
# Preview rejects zero or multiple selected sessions before any source data is loaded.
display(preview_session(config))
session_data = load_session(config, individual=INDIVIDUAL)
display(session_data.loaded.stream_coverage)
raw_pose = session_data.raw_pose
pose = prepare_pose(
    raw_pose, confidence_threshold=CONFIDENCE_THRESHOLD,
    max_gap_frames=MAX_GAP_FRAMES, smoothing_window=SMOOTHING_WINDOW,
)
position = pose.position.sel(individual=INDIVIDUAL, drop=True)
if TRACKING_KEYPOINT not in position.keypoint:
    raise ValueError(f"Choose TRACKING_KEYPOINT from {position.keypoint.values.tolist()}")
point = position.sel(keypoint=TRACKING_KEYPOINT, drop=True)
point.attrs["units"] = "px"
trials = session_data.trials.sort_values("start_time").reset_index(drop=True)
events = session_data.events
if trials.empty:
    raise ValueError("No parsed trials are available in this session.")
display(trials)
for note in qc_annotations(trials, events).notes:
    print(note)
print("Keypoints:", position.keypoint.values.tolist())
print("Valid tracked-point frames:", int(point.notnull().all("space").sum()),
      "/", point.sizes["time"])

if WINDOW is None:
    window_start = max(float(point.time.min()), float(trials.start_time.iloc[0]))
    WINDOW = (window_start, min(window_start + 60, float(point.time.max())))
if not np.isfinite(WINDOW).all() or WINDOW[0] >= WINDOW[1]:
    raise ValueError("WINDOW must be a finite increasing interval overlapping the pose.")
if not bool(((point.time >= WINDOW[0]) & (point.time <= WINDOW[1])).any()):
    raise ValueError("WINDOW contains no recorded pose samples; choose a window on the session clock.")
selected_rows = [i for i in TRIAL_ROWS if 0 <= i < len(trials)]
if len(selected_rows) != len(TRIAL_ROWS):
    print("Trial rows outside this session were omitted:", sorted(set(TRIAL_ROWS) - set(selected_rows)))
if not selected_rows:
    raise ValueError("TRIAL_ROWS must select at least one displayed trial row.")

## Allocentric heading and head direction relative to the body

**Allocentric heading** is the head's angle from a fixed reference axis. Set
`REFERENCE_VECTOR` to the arena reference expressed in the original image
coordinates. With the default (1, 0), zero points right in the camera image.

This first template interprets **egocentric head direction** as head orientation
relative to the body's forward axis (`BODY_BACK` → `BODY_FRONT`). Confirm this
definition for the paper. A target's bearing relative to the head is a different
measurement and would use movement's ROI-angle functions with a specified target.
The body vector is assembled from the two chosen positions; all angle calculations
use movement. Ear-based head direction assumes symmetric left/right labels.

Both angles span ±π. With +y down, positive angles are clockwise. Points are
plotted without connecting lines so crossings of the circular boundary do not
appear as large excursions through zero.

In [ ]:
allocentric_heading = kin.compute_forward_vector_angle(
    position, left_keypoint=LEFT_EAR, right_keypoint=RIGHT_EAR,
    reference_vector=REFERENCE_VECTOR, camera_view=CAMERA_VIEW,
)
head = kin.compute_head_direction_vector(
    position, left_keypoint=LEFT_EAR, right_keypoint=RIGHT_EAR,
    camera_view=CAMERA_VIEW,
)
body_forward = (position.sel(keypoint=BODY_FRONT, drop=True)
                - position.sel(keypoint=BODY_BACK, drop=True))
head_relative_to_body = compute_signed_angle_2d(body_forward, head)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, layout="constrained")
for index, (ax, heading, label) in enumerate(zip(
    axes, [allocentric_heading, head_relative_to_body],
    ["Allocentric head heading", "Head relative to body"],
)):
    visible = heading.sel(time=slice(*WINDOW))
    ax.plot(visible.time, visible, linestyle="none", marker=".", ms=2.5, color="#5b4b9a")
    ax.set_ylim(-np.pi, np.pi)
    ax.set_yticks([-np.pi, 0, np.pi], ["−π", "0", "π"])
    ax.set_ylabel(f"{label} (rad)")
    annotate_qc_timeseries(ax, trials, events, window=WINDOW, legend=index == 0)
axes[-1].set_xlabel("Session time (s)")
plt.show()

## Occupancy: whole session, outbound, and inbound

`movement.plots.plot_occupancy` reports **valid position samples per spatial bin**.
Counts are not dwell-time seconds, particularly if frame intervals are irregular.
Phase panels use only trials with known valid target-trigger boundaries. The
whole-session panel also includes inter-trial periods and trials without those
boundaries. Phases use half-open intervals, so a target-trigger sample belongs
to inbound and the end sample cannot be counted twice across adjacent trials.

All panels share spatial bin edges and the same colour limits; differing phase
durations therefore affect the raw counts. No probability or time normalisation
is applied. Missing tracked positions are excluded by movement.

In [ ]:
finite_point = point.where(np.isfinite(point).all("space"), drop=True)
if finite_point.sizes["time"] == 0:
    raise ValueError("No valid positions for occupancy.")
if ARENA_RANGE is None:
    ARENA_RANGE = tuple((float(finite_point.sel(space=axis).min()),
                         float(finite_point.sel(space=axis).max())) for axis in ["x", "y"])
if any(low >= high for low, high in ARENA_RANGE):
    raise ValueError("ARENA_RANGE needs a nonzero range on both axes.")

outbound_mask = np.zeros(point.sizes["time"], dtype=bool)
inbound_mask = np.zeros(point.sizes["time"], dtype=bool)
phase_trial_count = 0
for trial in trials.itertuples():
    start, split, end = float(trial.start_time), float(trial.tz_triggered_time), float(trial.end_time)
    if np.isfinite([start, split, end]).all() and start < split < end:
        outbound_mask |= (point.time.values >= start) & (point.time.values < split)
        inbound_mask |= (point.time.values >= split) & (point.time.values < end)
        phase_trial_count += 1
print(f"Known phase boundaries: {phase_trial_count}/{len(trials)} trials")
phase_points = [point, point.isel(time=outbound_mask), point.isel(time=inbound_mask)]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
occupancy = {}
colour_limit = 1.0
for ax, label, data in zip(axes, ["Whole session", "Outbound", "Inbound"], phase_points):
    if data.sizes["time"] == 0 or not bool(np.isfinite(data).all("space").any()):
        ax.text(0.5, 0.5, "No valid samples", ha="center", transform=ax.transAxes)
    else:
        _, _, histogram = plot_occupancy(data, ax=ax, bins=OCCUPANCY_BINS,
                                         range=ARENA_RANGE, cmap="magma")
        occupancy[label] = histogram
        colour_limit = max(colour_limit, float(histogram["h"].max()))
        ax.collections[-1].colorbar.set_label("Valid position samples / bin")
    ax.set_title(label)
    ax.set_xlim(*ARENA_RANGE[0])
    ax.set_ylim(ARENA_RANGE[1][1], ARENA_RANGE[1][0])
    ax.set_aspect("equal")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
for ax in axes:
    if ax.collections:
        ax.collections[-1].set_clim(0, colour_limit)
plt.show()

API references: [head-direction example](https://movement.neuroinformatics.dev/latest/examples/compute_head_direction.html),
[forward-vector angle](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_forward_vector_angle.html),
[signed angle](https://movement.neuroinformatics.dev/latest/api/movement.utils.vector.compute_signed_angle_2d.html),
[occupancy](https://movement.neuroinformatics.dev/latest/api/movement.plots.plot_occupancy.html).